# 🩺 Medical RAG Assistant
### Retrieval-Augmented Generation on Medical PDFs — Interview Ready

---

**What this notebook does:**
Upload any medical PDF → ask natural language questions → get answers grounded in the document with page citations.

**The RAG idea in one sentence:**
> Instead of asking the LLM from memory, we *retrieve* the relevant text from the document first, then ask the LLM to answer *only from that text*.

**Tech Stack:**
| Step | Tool |
|---|---|
| PDF Loading | `langchain-community` PyPDFLoader |
| Text Chunking | `langchain` RecursiveCharacterTextSplitter |
| Embeddings | `sentence-transformers` — all-MiniLM-L6-v2 (free, CPU) |
| Vector Database | `chromadb` (local, no server needed) |
| LLM | Google Gemini 2.5 Flash via `langchain-google-genai` |
| RAG Chain | `langchain` RetrievalQA |

✅ Runs fully in **Google Colab** — no local setup needed.

---
## Step 1 — Install Dependencies

Run this once at the start of your Colab session.  
These are the only packages this project needs.

In [2]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-google-genai \
    chromadb \
    sentence-transformers \
    pypdf \
    google-generativeai

print("✅ All packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 69.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7

---
## Step 2 — Imports

All imports in one place — easy to see every tool being used.

In [5]:
import os

# PDF loading
from langchain_community.document_loaders import PyPDFLoader

# Text splitting
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Embeddings — local model, no API key needed
from langchain_community.embeddings import HuggingFaceEmbeddings

# Vector database
from langchain_community.vectorstores import Chroma

# LLM — Google Gemini
from langchain_google_genai import ChatGoogleGenerativeAI

# RAG chain — ties retriever + LLM together
from langchain_classic.chains import RetrievalQA

print("✅ Imports successful")

✅ Imports successful


---
## Step 3 — Upload Your Medical PDF

Run this cell → a file picker will appear → choose your PDF.  
Works directly in Google Colab — no file paths to configure.

In [6]:
from google.colab import files

print("📁 Select your medical PDF file...")
uploaded = files.upload()

# Get the filename of the uploaded file
pdf_filename = list(uploaded.keys())[0]

print(f"\n✅ Uploaded: {pdf_filename}")
print(f"   Size: {len(uploaded[pdf_filename]) / 1024:.1f} KB")

📁 Select your medical PDF file...


Saving Comprehensive Medical Knowledge Base.pdf to Comprehensive Medical Knowledge Base.pdf

✅ Uploaded: Comprehensive Medical Knowledge Base.pdf
   Size: 62.5 KB


---
## Step 4 — Load the PDF

**PyPDFLoader** reads the PDF and gives us one `Document` object per page.  
Each `Document` contains the page text + metadata (filename, page number).

In [7]:
loader = PyPDFLoader(pdf_filename)
pages = loader.load()

print(f"✅ PDF loaded successfully")
print(f"   Pages found: {len(pages)}")
print(f"\n--- Preview of Page 1 ---")
print(pages[0].page_content[:500])

✅ PDF loaded successfully
   Pages found: 16

--- Preview of Page 1 ---
Comprehensive Medical Knowledge Base for RAG
Testing
Introduction to Human Health
Human health is influenced by genetics, nutrition, environment, exercise, sleep, mental well-being, and
healthcare access. The human body contains multiple interconnected systems including the cardiovascular
system, respiratory system, digestive system, endocrine system, nervous system, immune system, and
musculoskeletal system.
A healthy adult typically maintains normal blood pressure between 90/60 mmHg and 120/80


---
## Step 5 — Split Text into Chunks

**Why do we chunk?**  
LLMs have a limited context window — we can't send the entire PDF at once.  
We split the document into small, overlapping chunks. When a question comes in,  
we only send the *most relevant* chunks to the LLM.

- `chunk_size = 1000` → each chunk is ~1000 characters  
- `chunk_overlap = 200` → chunks overlap by 200 characters so context isn't lost at boundaries

In [8]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)

chunks = splitter.split_documents(pages)

print(f"✅ Document split into {len(chunks)} chunks")
print(f"\n--- Sample Chunk ---")
print(f"From page: {chunks[0].metadata.get('page', 0) + 1}")
print(f"Text: {chunks[0].page_content[:300]}...")

✅ Document split into 17 chunks

--- Sample Chunk ---
From page: 1
Text: Comprehensive Medical Knowledge Base for RAG
Testing
Introduction to Human Health
Human health is influenced by genetics, nutrition, environment, exercise, sleep, mental well-being, and
healthcare access. The human body contains multiple interconnected systems including the cardiovascular
system, re...


---
## Step 6 — Load the Embedding Model

**What is an embedding?**  
An embedding converts text into a list of numbers (a vector) that captures *meaning*.  
Sentences with similar meaning get similar vectors.

**Model: `all-MiniLM-L6-v2`**
- Produces 384-dimensional vectors
- Free, runs on CPU, no API key needed
- Downloads automatically (~90 MB, cached after first run)

In [9]:
print("⏳ Loading embedding model (downloads ~90MB on first run)...")

embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

# Quick test — embed one sentence
test_vector = embedding_model.embed_query("What are the symptoms?")

print(f"✅ Embedding model loaded")
print(f"   Vector dimensions: {len(test_vector)}")

⏳ Loading embedding model (downloads ~90MB on first run)...


/tmp/ipykernel_2207/2325195177.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded
   Vector dimensions: 384


---
## Step 7 — Create the Vector Database (ChromaDB)

**What happens here?**  
Every chunk gets converted into an embedding vector and stored in ChromaDB.  
This is the "index" — we only do this once.

**At query time:**  
The user's question is also embedded → ChromaDB finds the chunks whose vectors are  
closest to the question vector → those are the most relevant chunks.

In [10]:
print(f"⏳ Embedding {len(chunks)} chunks and storing in ChromaDB...")

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
)

print(f"✅ Vector database created")
print(f"   Total vectors stored: {vectordb._collection.count()}")

⏳ Embedding 17 chunks and storing in ChromaDB...
✅ Vector database created
   Total vectors stored: 17


---
## Step 8 — Create the Retriever

The retriever is the search interface over the vector database.  
Given any query, it returns the top-K most relevant chunks.

`k=3` → fetch the 3 most relevant chunks for each question.

In [11]:
retriever = vectordb.as_retriever(search_kwargs={"k": 3})

# Test the retriever with a sample question
test_question = "What are the main topics covered in this document?"
retrieved_docs = retriever.invoke(test_question)

print(f"✅ Retriever ready — fetches top 3 chunks per question")
print(f"\n🔍 Test retrieval for: '{test_question}'")
for i, doc in enumerate(retrieved_docs, 1):
    page = doc.metadata.get("page", 0) + 1
    print(f"\n  Chunk {i} (Page {page}):")
    print(f"  {doc.page_content[:200]}...")

✅ Retriever ready — fetches top 3 chunks per question

🔍 Test retrieval for: 'What are the main topics covered in this document?'

  Chunk 1 (Page 16):
  Public Health
Public health focuses on disease prevention and community health improvement.
Public Health Measures
Vaccination programs
Clean water supply
Sanitation
Health education
Conclusion
Health...

  Chunk 2 (Page 12):
  Antibiotics are ineffective against viral infections.
Analgesics
Analgesics reduce pain.
Examples:
Paracetamol
Ibuprofen
Overuse of NSAIDs may damage the stomach lining.
Medical Imaging
MRI
Magnetic R...

  Chunk 3 (Page 4):
  Chest pain
Risk Groups
Elderly
Infants
Immunocompromised individuals
Treatment
Antibiotics
Antiviral medications
Oxygen therapy
Hydration
Endocrine Disorders
Diabetes Mellitus
Diabetes mellitus is a c...


---
## Step 9 — Add Your Gemini API Key

Get a free key at: **https://aistudio.google.com/app/apikey**

Paste it below when prompted.

In [14]:
from getpass import getpass

GEMINI_API_KEY = getpass("🔑 Enter your Gemini API Key: ")

# Set as environment variable (used by langchain-google-genai internally)
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

print("✅ API key saved")

🔑 Enter your Gemini API Key: ··········
✅ API key saved


---
## Step 10 — Initialize the Gemini LLM

**Why Gemini 2.5 Flash?**
- Fast response time
- Large context window (fits multiple retrieved chunks easily)
- Free tier available

**Why `temperature=0`?**  
For medical QA, we want factual and deterministic answers — not creative ones.  
Temperature 0 means the model always picks the most likely (safest) answer.

In [15]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0,           # 0 = deterministic/factual, 1 = creative
    google_api_key=GEMINI_API_KEY,
)

print("✅ Gemini 2.5 Flash initialized")
print("   temperature=0 → factual, grounded answers")

✅ Gemini 2.5 Flash initialized
   temperature=0 → factual, grounded answers


---
## Step 11 — Build the RetrievalQA Chain

**This is where RAG comes together.**

`RetrievalQA` is a LangChain chain that does everything automatically:
1. Takes your question
2. Retrieves the top-K relevant chunks from ChromaDB
3. Injects those chunks as context into the prompt
4. Sends it to Gemini
5. Returns the grounded answer

`chain_type="stuff"` = stuffs all retrieved chunks directly into the prompt (simplest approach, works well with small k).

`return_source_documents=True` = also returns which chunks were used, so we can show citations.

In [16]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",              # Simplest: stuff all chunks into one prompt
    retriever=retriever,
    return_source_documents=True,    # So we can show where answers came from
)

print("✅ RAG chain ready")
print("   Pipeline: Question → Retriever → ChromaDB → Gemini → Answer + Sources")

✅ RAG chain ready
   Pipeline: Question → Retriever → ChromaDB → Gemini → Answer + Sources


---
## Step 12 — Ask a Question!

Change the question below and run the cell.  
The answer is generated purely from your uploaded PDF — no hallucinations from outside knowledge.

In [17]:
# ✏️ Change this question to anything about your document
question = "What are the main symptoms and diagnostic criteria mentioned?"

# Run the RAG chain
result = qa_chain.invoke({"query": question})

# --- Print the answer ---
print("=" * 60)
print("❓ QUESTION")
print("=" * 60)
print(question)

print("\n" + "=" * 60)
print("💡 ANSWER")
print("=" * 60)
print(result["result"])

# --- Print source citations ---
print("\n" + "=" * 60)
print("📚 SOURCES (chunks used to generate this answer)")
print("=" * 60)
for i, doc in enumerate(result["source_documents"], 1):
    page = doc.metadata.get("page", 0) + 1
    print(f"\n[Source {i}] — Page {page}")
    print("-" * 40)
    print(doc.page_content[:400] + "...")

❓ QUESTION
What are the main symptoms and diagnostic criteria mentioned?

💡 ANSWER
Based on the provided text, the main symptoms mentioned for various conditions are:

**Hypertension:**
*   Headache
*   Dizziness
*   Blurred vision
*   Chest pain
*   Fatigue

**Diabetes Mellitus:**
*   Excessive thirst
*   Frequent urination
*   Fatigue
*   Weight loss
*   Blurred vision

**Gastritis (implied by symptoms before Peptic Ulcer Disease):**
*   Nausea
*   Vomiting
*   Upper abdominal pain
*   Bloating

**Peptic Ulcer Disease:**
*   Burning stomach pain
*   Bloating
*   Heartburn
*   Nausea

**Tuberculosis:**
*   Chronic cough
*   Fever

The provided text does not mention any specific diagnostic criteria for these conditions.

📚 SOURCES (chunks used to generate this answer)

[Source 1] — Page 1
----------------------------------------
consistently elevated. Hypertension may not show symptoms initially but can significantly increase the risk
of heart disease, stroke, kidney disease, and heart

---
## Step 13 — Ask Questions

Run this cell to ask multiple questions in a row.  
Type `quit` to stop.

In [18]:
print("🩺 Medical RAG — Interactive Mode")
print("   Ask questions about your document. Type 'quit' to stop.")
print("-" * 60)

while True:
    question = input("\n❓ Your question: ").strip()

    if not question:
        continue

    if question.lower() in ("quit", "exit", "q"):
        print("👋 Done.")
        break

    result = qa_chain.invoke({"query": question})

    print("\n💡 Answer:")
    print(result["result"])

    print("\n📚 Sources:")
    for doc in result["source_documents"]:
        page = doc.metadata.get("page", 0) + 1
        print(f"  • Page {page}: {doc.page_content[:120]}...")

    print("-" * 60)

🩺 Medical RAG — Interactive Mode
   Ask questions about your document. Type 'quit' to stop.
------------------------------------------------------------

❓ Your question: i am getting headache severely 

💡 Answer:
Based on the information provided, a severe headache can be a symptom of **Migraine**.

It is also listed as a symptom of **Hypertension**, though it states that hypertension "may not show symptoms initially".

If you are experiencing a severe headache, it's important to seek medical advice.

📚 Sources:
  • Page 5: Treatment
Insulin therapy
Metformin
Dietary management
Exercise
Prevention
Weight management
Healthy eating
Physical act...
  • Page 11: Emergency Medicine
Stroke
A stroke occurs when blood supply to the brain is interrupted.
Symptoms
Facial drooping
Arm we...
  • Page 1: Comprehensive Medical Knowledge Base for RAG
Testing
Introduction to Human Health
Human health is influenced by genetics...
------------------------------------------------------------

❓ Your que

****

In [20]:
from datasets import Dataset

eval_data = {
    "question": [
        "What are the symptoms of hypertension?",
        "What are the types of diabetes?",
        "What is tuberculosis?",
        "What are common medical imaging techniques?"
    ],

    "ground_truth": [
        "Common symptoms include headache, dizziness, blurred vision, chest pain, and fatigue.",

        "The main types are Type 1 Diabetes and Type 2 Diabetes.",

        "Tuberculosis is a bacterial infectious disease that mainly affects the lungs.",

        "Common imaging techniques include MRI, CT scan, X-ray, and ultrasound."
    ]
}

eval_dataset = Dataset.from_dict(eval_data)

print("✅ Evaluation dataset ready")

✅ Evaluation dataset ready


In [27]:
answers = []
contexts = []

print("⏳ Generating answers...")

for question in eval_dataset["question"]:

    # Retrieve relevant chunks
    retrieved_docs = retriever.invoke(question)

    # Store contexts
    contexts.append(
        [doc.page_content for doc in retrieved_docs]
    )

    # Generate answer
    response = qa_chain.invoke({"query": question})

    # Append answer inside the loop
    answers.append(response["result"])

print("✅ Answers generated")

print(f"Questions: {len(eval_dataset['question'])}")
print(f"Answers: {len(answers)}")
print(f"Contexts: {len(contexts)}")

⏳ Generating answers...


ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 40.426611178s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

In [22]:
eval_dataset = eval_dataset.remove_columns(
    [col for col in eval_dataset.column_names if col in ["answer", "contexts"]]
)

eval_dataset = eval_dataset.add_column("answer", answers)

eval_dataset = eval_dataset.add_column("contexts", contexts)

print("✅ Dataset updated")

ValueError: Failed to concatenate on axis=1 because tables don't have the same number of rows

In [ ]:
from ragas import evaluate

from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

print("⏳ Running RAG evaluation...")

result = evaluate(
    dataset=eval_dataset,

    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]
)

print("✅ Evaluation completed")

In [ ]:
print(result)
df = result.to_pandas()
df

In [ ]:
print("📊 Metric Explanations")

print("\n1. Faithfulness")
print("Checks whether the answer is supported by retrieved context.")

print("\n2. Answer Relevancy")
print("Checks how relevant the answer is to the question.")

print("\n3. Context Precision")
print("Checks whether retrieved chunks are actually useful.")

print("\n4. Context Recall")
print("Checks whether important information was retrieved.")